# 07 — Recommendation Engine Prototype

**Purpose:** Implement every pure function from `RECOMMENDATION_PROTOCOL.md` v2.0 verbatim,
then run synthetic scenarios to validate the decision logic before handing the spec to dev.

**What this notebook is:**
- A runnable prototype of the spec — no DB, no FastAPI, no Garmin connection
- A synthetic test suite: each scenario documents a state, expected output, and actual output
- A spec-gap log: anything that produces unexpected results is flagged for spec revision

**What this notebook is NOT:**
- Production code (it lives here, not in `intelligence/`)
- A substitute for the dev implementation

**Sections:**
1. Signal computation functions (§3)
2. Safety gates (§4)
3. Session selection (§5)
4. Running prescription (§8)
5. Strength prescription (§7)
6. Full pipeline assembly
7. Synthetic scenario suite
8. Findings & spec gaps

In [34]:
from math import exp
from statistics import mean, stdev
from itertools import takewhile
from datetime import date, datetime, timedelta
from dataclasses import dataclass, field
from typing import Any
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# ── Spec constants ────────────────────────────────────────────────────────────
STANDARD_DISCLAIMER = (
    "This is a training load suggestion based on your personal data. "
    "It is not medical advice. If you experience pain, illness, or unusual "
    "fatigue, consult a qualified healthcare provider before training."
)

SPEC_GAPS: list[dict] = []   # populated as we find issues

def flag_gap(section: str, description: str, severity: str = 'minor'):
    """Record a spec gap for the findings section."""
    SPEC_GAPS.append({'section': section, 'description': description, 'severity': severity})
    print(f'  ⚠ SPEC GAP [{severity.upper()}] §{section}: {description}')

print('Imports OK')

Imports OK


## §3.2 — TRIMP

In [35]:
def compute_trimp(
    duration_min: float,
    avg_hr:       float,
    resting_hr:   float,
    max_hr:       float,
    sex:          str,      # 'male' | 'female' | 'prefer_not_to_say'
) -> float:
    HRr = (avg_hr - resting_hr) / (max_hr - resting_hr)
    HRr = max(0.0, min(1.0, HRr))
    k, y = (0.86, 1.67) if sex == 'female' else (0.64, 1.92)
    return duration_min * HRr * k * exp(y * HRr)

# ── Sanity checks ─────────────────────────────────────────────────────────────
# Easy Z2 run: 60 min, avg HR 145, RHR 50, max 190 → HRr ≈ 0.68
t_easy = compute_trimp(60, 145, 50, 190, 'male')
# Threshold run: 45 min, avg HR 170 → HRr ≈ 0.86
t_threshold = compute_trimp(45, 170, 50, 190, 'male')
# Rest: 0 min
t_rest = compute_trimp(0, 130, 50, 190, 'male')

print(f'Z2 run (60min, HR145):        TRIMP = {t_easy:.1f}')
print(f'Threshold run (45min, HR170): TRIMP = {t_threshold:.1f}')
print(f'Rest:                         TRIMP = {t_rest:.1f}')
print()

# Female vs male at same effort
t_female = compute_trimp(60, 155, 55, 195, 'female')
t_male   = compute_trimp(60, 155, 55, 195, 'male')
print(f'Same session — female TRIMP: {t_female:.1f}, male TRIMP: {t_male:.1f}')
print(f'Female/male ratio: {t_female/t_male:.2f} (expected ~1.0; sex coefficients shift curve shape, not magnitude)')

assert t_threshold > t_easy, 'Threshold should have higher TRIMP than easy'
assert t_rest == 0.0, 'Rest should be 0'
print('\nTRIMP: all checks passed')

Z2 run (60min, HR145):        TRIMP = 95.9
Threshold run (45min, HR170): TRIMP = 128.0
Rest:                         TRIMP = 0.0

Same session — female TRIMP: 121.5, male TRIMP: 108.1
Female/male ratio: 1.12 (expected ~1.0; sex coefficients shift curve shape, not magnitude)

TRIMP: all checks passed


## §3.3 — ATL / CTL / TSB / ACWR

In [36]:
def _ctl_at_offset(
    trimp_series: list[tuple],
    today: date,
    days: int,
) -> float | None:
    """Recompute CTL as if today were `days` ago."""
    cutoff = today - timedelta(days=days)
    subset = [(d, t) for d, t in trimp_series if d <= cutoff]
    if not subset:
        return None
    TAU_CTL = 42.0
    ctl = 0.0
    prev = subset[0][0]
    for session_date, trimp in subset:
        gap = (session_date - prev).days
        ctl = ctl * exp(-gap / TAU_CTL) + trimp * (1 - exp(-1 / TAU_CTL))
        prev = session_date
    return ctl


def compute_load_metrics(
    trimp_series: list[tuple],   # [(date, trimp), ...] sorted asc
    today: date,
) -> dict:
    TAU_ATL, TAU_CTL = 7.0, 42.0
    atl = ctl = 0.0
    if not trimp_series:
        return {'atl': 0.0, 'ctl': 0.0, 'tsb': 0.0, 'acwr': None, 'ramp_rate': None}
    prev = trimp_series[0][0]
    for session_date, trimp in trimp_series:
        gap = (session_date - prev).days
        atl = atl * exp(-gap / TAU_ATL) + trimp * (1 - exp(-1 / TAU_ATL))
        ctl = ctl * exp(-gap / TAU_CTL) + trimp * (1 - exp(-1 / TAU_CTL))
        prev = session_date
    tsb  = ctl - atl
    acwr = round(atl / ctl, 3) if ctl >= 1.0 else None
    ctl_7d = _ctl_at_offset(trimp_series, today, days=7)
    ramp   = ((ctl - ctl_7d) / max(ctl_7d, 1.0)) * 100 if ctl_7d is not None else None
    return {'atl': round(atl, 2), 'ctl': round(ctl, 2), 'tsb': round(tsb, 2),
            'acwr': acwr, 'ramp_rate': round(ramp, 1) if ramp is not None else None}


# ── Scenario: 90-day buildup then spike ───────────────────────────────────────
def make_training_block(start: date, days: int, trimp_per_day: float,
                        days_off_per_week: int = 1) -> list[tuple]:
    """Generate a uniform training block."""
    series = []
    d = start
    week_day = 0
    for _ in range(days):
        if week_day < (7 - days_off_per_week):
            series.append((d, trimp_per_day))
        d += timedelta(days=1)
        week_day = (week_day + 1) % 7
    return series

today = date(2026, 5, 13)
base_start = today - timedelta(days=90)

# 90 days of moderate base (TRIMP ~50/session)
base_block  = make_training_block(base_start, 83, 50)
# Sudden spike: last 7 days at TRIMP ~120
spike_block = make_training_block(today - timedelta(days=7), 7, 120, days_off_per_week=0)
series_spike = base_block + spike_block

metrics_base  = compute_load_metrics(base_block, today)
metrics_spike = compute_load_metrics(series_spike, today)

print('=== Baseline (90 days moderate) ===')
print(f'  ATL={metrics_base["atl"]:.1f}  CTL={metrics_base["ctl"]:.1f}  TSB={metrics_base["tsb"]:.1f}  ACWR={metrics_base["acwr"]}  Ramp={metrics_base["ramp_rate"]}%')
print()
print('=== After 7-day spike ===')
print(f'  ATL={metrics_spike["atl"]:.1f}  CTL={metrics_spike["ctl"]:.1f}  TSB={metrics_spike["tsb"]:.1f}  ACWR={metrics_spike["acwr"]}  Ramp={metrics_spike["ramp_rate"]}%')
print()

# ACWR thresholds from spec
acwr = metrics_spike['acwr']
if acwr is not None:
    label = ('critical' if acwr > 1.8 else 'danger' if acwr > 1.5 else
              'caution' if acwr > 1.3 else 'optimal' if acwr >= 0.8 else 'under_training')
    print(f'  ACWR zone: {label}')
    if acwr < 1.3:
        print('  NOTE: spike block may not be large enough to cross caution threshold — adjust TRIMP if testing gates')

=== Baseline (90 days moderate) ===
  ATL=45.5  CTL=37.5  TSB=-8.0  ACWR=1.215  Ramp=0.0%

=== After 7-day spike ===
  ATL=92.6  CTL=50.2  TSB=-42.5  ACWR=1.846  Ramp=27.2%

  ACWR zone: critical


## §3.4 — HRV Status

In [ ]:
def establish_hrv_baseline(
    hrv_series:      list[float],         # HRV readings sorted asc by date
    preceding_trimp: list[float | None],  # TRIMP of the preceding day; None = rest day
    easy_threshold:  float = 50.0,        # TRIMP ≤ this counts as a clean reading
    min_clean:       int   = 14,
) -> tuple[float, float] | None:
    """
    Compute personal HRV baseline using only readings that follow rest or easy days.

    Rationale: HRV measured after a high-TRIMP session reflects acute training stress,
    not stable resting baseline. Pooling all readings contaminates the mean during
    heavy blocks — suppressed days look normal. By filtering to low-load preceding days
    (rest or easy Z2), clean readings accumulate across the full training year without
    requiring a dedicated "baseline week".

    easy_threshold=50: roughly 60–70 min of easy Z2 work, not stressful enough to
    depress next-morning HRV. Adjust per athlete if their aerobic TRIMP runs higher.
    """
    if len(hrv_series) != len(preceding_trimp):
        return None
    clean = [h for h, t in zip(hrv_series, preceding_trimp)
             if t is None or t <= easy_threshold]
    if len(clean) < min_clean:
        return None
    m, s = mean(clean), stdev(clean)
    return (round(m, 2), round(s, 2)) if s >= 0.5 else None


def compute_hrv_status(
    hrv_series:    list[float],
    personal_mean: float,
    personal_sd:   float,
) -> dict:
    """
    hrv_series: overnight HRV sorted asc by date, most recent last.
    personal_mean / personal_sd: from user_profile — set by establish_hrv_baseline().
    Returns 'no_data' when personal_sd < 0.5 (baseline not yet established).
    """
    if len(hrv_series) < 2 or personal_sd < 0.5:
        return {'status': 'no_data', 'z': None, 'consecutive_suppressed': 0}
    z = (hrv_series[-1] - personal_mean) / personal_sd
    consec = sum(1 for _ in takewhile(
        lambda v: (v - personal_mean) / personal_sd < -1.0,
        reversed(hrv_series[:-1])
    ))
    status = ('elevated'       if z >  1.0 else
              'normal'         if z > -1.0 else
              'suppressed'     if z > -1.5 else
              'very_suppressed')
    return {'status': status, 'z': round(z, 2), 'consecutive_suppressed': consec}


def _make_hrv_series(
    personal_mean: float,
    personal_sd: float,
    n_history: int,
    tail: list[float],
    consec_suppressed: int = 0,
    seed: int = 42,
) -> list[float]:
    rng = np.random.default_rng(seed)
    history = list(rng.normal(personal_mean, personal_sd, n_history))
    for i in range(consec_suppressed):
        history[-(i + 1)] = personal_mean - 1.3 * personal_sd
    return history + tail


PM, PSD = 80, 8   # realistic endurance athlete baseline

# ── establish_hrv_baseline ────────────────────────────────────────────────────

# Test 1: all clean days → baseline established
rng = np.random.default_rng(0)
hrv_all_clean = list(rng.normal(PM, PSD, 20))
bl = establish_hrv_baseline(hrv_all_clean, [None] * 20)
print(f'All rest days (n=20): mean={bl[0]}, sd={bl[1]}')
assert bl is not None
assert abs(bl[0] - PM) < 3 and bl[1] > 0.5

# Test 2: fewer than min_clean qualifying readings → None
hrv_mostly_hard = list(rng.normal(PM, PSD, 30))
preceding_mostly_hard = [120.0] * 20 + [None] * 10   # only 10 clean readings
bl_short = establish_hrv_baseline(hrv_mostly_hard, preceding_mostly_hard)
print(f'Only 10 clean days (need 14): {bl_short}')
assert bl_short is None

# Test 3: dirty readings are correctly excluded from the mean
# True resting HRV = PM (80). Post-hard-workout readings artificially suppressed to 60.
hrv_mixed    = [60.0] * 20 + [PM] * 14       # 20 hard-day, 14 rest-day
preceding_mixed = [150.0] * 20 + [None] * 14
bl_clean = establish_hrv_baseline(hrv_mixed, preceding_mixed)
print(f'20 dirty + 14 clean: mean={bl_clean[0]} (expected ~{PM}, not ~60)')
assert bl_clean is not None
assert abs(bl_clean[0] - PM) < 5, f'Dirty readings contaminated baseline: {bl_clean[0]}'

print('establish_hrv_baseline: all checks passed\n')

# ── compute_hrv_status ────────────────────────────────────────────────────────

# Scenario A: elevated (z = +1.3)
res_a = compute_hrv_status(_make_hrv_series(PM, PSD, 30, [PM + 1.3 * PSD]), PM, PSD)
print(f'Elevated HRV:  {res_a}')
assert res_a['status'] == 'elevated'

# Scenario B: suppressed (z = -1.26, inside -1.5 < z < -1.0)
res_b = compute_hrv_status(_make_hrv_series(PM, PSD, 30, [PM - 1.26 * PSD]), PM, PSD)
print(f'Suppressed HRV: {res_b}')
assert res_b['status'] == 'suppressed'

# Scenario C: 5 consecutive suppressed days → triggers Gate 2
res_c = compute_hrv_status(
    _make_hrv_series(PM, PSD, 30, [PM - 1.26 * PSD], consec_suppressed=5),
    PM, PSD,
)
print(f'Consecutive suppressed: {res_c}')
assert res_c['consecutive_suppressed'] >= 5

# Scenario D: no baseline (sd=0) → no_data
res_d = compute_hrv_status([PM, PM - 1, PM + 1], personal_mean=PM, personal_sd=0.0)
print(f'No baseline:   {res_d}')
assert res_d['status'] == 'no_data'

print('\nHRV status: all checks passed')

## §3.5 — Sleep Readiness

In [38]:
def compute_sleep_readiness(
    sleep_score:      int   | None,
    hrv_z:            float | None,
    body_battery:     int   | None,
    sleep_duration_h: float | None,
) -> str:  # 'high' | 'moderate' | 'low' | 'no_data'
    available = [x for x in [sleep_score, hrv_z, body_battery] if x is not None]
    if not available:
        return 'no_data'
    score = 0
    if sleep_score      is not None: score += (2 if sleep_score  >= 75 else 1 if sleep_score  >= 55 else 0)
    if hrv_z            is not None: score += (2 if hrv_z        >  0.0 else 1 if hrv_z        > -1.0 else 0)
    if body_battery     is not None: score += (2 if body_battery >= 60  else 1 if body_battery >= 35  else 0)
    if sleep_duration_h is not None: score += (1 if sleep_duration_h >= 7.5 else -1 if sleep_duration_h < 6.0 else 0)
    max_score = 2 * len(available) + (1 if sleep_duration_h is not None else 0)
    ratio  = score / max(max_score, 1)
    result = 'high' if ratio >= 0.60 else 'moderate' if ratio >= 0.30 else 'low'
    # Hard cap: < 6h sleep cannot produce 'high' readiness regardless of other signals.
    # The -1 ratio penalty is insufficient when all other signals are excellent.
    if sleep_duration_h is not None and sleep_duration_h < 6.0 and result == 'high':
        result = 'moderate'
    return result


print('Sleep readiness scenarios:')
scenarios = [
    ('Great night',          dict(sleep_score=82, hrv_z=0.8,  body_battery=72, sleep_duration_h=8.0),  'high'),
    ('Decent night',         dict(sleep_score=65, hrv_z=0.1,  body_battery=45, sleep_duration_h=7.0),  'moderate'),
    ('Poor night',           dict(sleep_score=45, hrv_z=-1.8, body_battery=20, sleep_duration_h=5.5),  'low'),
    ('No data',              dict(sleep_score=None, hrv_z=None, body_battery=None, sleep_duration_h=None), 'no_data'),
    ('Only HRV',             dict(sleep_score=None, hrv_z=1.2,  body_battery=None, sleep_duration_h=None), 'high'),
    ('Short sleep override', dict(sleep_score=80, hrv_z=0.5, body_battery=65, sleep_duration_h=5.0), 'moderate'),
]

all_pass = True
for name, kwargs, expected in scenarios:
    result = compute_sleep_readiness(**kwargs)
    ok = '✓' if result == expected else '✗'
    if result != expected:
        all_pass = False
    print(f'  {ok} {name:<25} → {result:<10} (expected {expected})')

if all_pass:
    print('\nSleep readiness: all checks passed')
else:
    flag_gap('3.5', 'Sleep readiness scoring does not match expected outputs', 'minor')

Sleep readiness scenarios:
  ✓ Great night               → high       (expected high)
  ✓ Decent night              → moderate   (expected moderate)
  ✓ Poor night                → low        (expected low)
  ✓ No data                   → no_data    (expected no_data)
  ✓ Only HRV                  → high       (expected high)
  ✓ Short sleep override      → moderate   (expected moderate)

Sleep readiness: all checks passed


## §3.7 — Terrain Classification

In [39]:
def classify_terrain(gradient_series: list[float]) -> str:
    if len(gradient_series) < 10:
        return 'unknown'
    pct_graded = sum(1 for g in gradient_series if abs(g) > 3) / len(gradient_series)
    grad_range = max(gradient_series) - min(gradient_series)
    return 'trail' if (pct_graded > 0.25 or grad_range > 15) else 'road'


# Flat road run
road_gradients = list(np.random.default_rng(1).normal(0, 1.5, 500))
# Trail run with lots of elevation
trail_gradients = list(np.random.default_rng(2).uniform(-12, 18, 500))
# Too short
short_gradients = [1.0, 2.0, -1.0]

print(f'Flat road:  {classify_terrain(road_gradients)}')
print(f'Trail run:  {classify_terrain(trail_gradients)}')
print(f'Short data: {classify_terrain(short_gradients)}')

assert classify_terrain(road_gradients)   == 'road'
assert classify_terrain(trail_gradients)  == 'trail'
assert classify_terrain(short_gradients)  == 'unknown'
print('Terrain classification: all checks passed')

Flat road:  road
Trail run:  trail
Short data: unknown
Terrain classification: all checks passed


## §3.8 — Biomechanics Fatigue Index

In [40]:
from scipy.stats import linregress

@dataclass
class BiomechanicsBaseline:
    gct_mean_ms:          float
    gct_sd_ms:            float
    vertical_ratio_mean:  float
    vertical_ratio_sd:    float


def compute_fatigue_index(records_df: pd.DataFrame, baseline: BiomechanicsBaseline | None):
    if baseline is None or len(records_df) < 20:
        return None, {}
    slope, *_ = linregress(records_df['distance_km'], records_df['stance_time_ms'])
    stance_drift = slope / max(baseline.gct_mean_ms, 1)
    n   = len(records_df)
    cv1 = records_df[:n//2]['cadence_rpm'].std() / max(records_df[:n//2]['cadence_rpm'].mean(), 1)
    cv2 = records_df[n//2:]['cadence_rpm'].std() / max(records_df[n//2:]['cadence_rpm'].mean(), 1)
    cv_r = cv2 / max(cv1, 0.001)
    vr_z = ((records_df['vertical_ratio'].mean() - baseline.vertical_ratio_mean)
            / max(baseline.vertical_ratio_sd, 0.001))
    index = abs(stance_drift) * 0.50 + (cv_r - 1.0) * 0.30 + abs(vr_z) * 0.20
    return round(index, 3), {
        'stance_drift':      round(stance_drift, 4),
        'cadence_cv_ratio':  round(cv_r, 3),
        'vertical_ratio_z':  round(vr_z, 3),
    }


def _make_run_records(n: int, stance_slope: float = 0, cadence_cv_increase: float = 1.0,
                      vr_offset: float = 0.0, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    distances = np.linspace(0, 10, n)
    stance    = 260 + stance_slope * distances + rng.normal(0, 3, n)
    # Cadence: increase CV in second half
    cadence   = rng.normal(180, 3 * cadence_cv_increase, n)
    cadence[n//2:] = rng.normal(180, 3 * cadence_cv_increase * 1.5, n - n//2)
    vr        = 8.5 + vr_offset + rng.normal(0, 0.3, n)
    return pd.DataFrame({'distance_km': distances, 'stance_time_ms': stance,
                         'cadence_rpm': cadence, 'vertical_ratio': vr})


baseline = BiomechanicsBaseline(gct_mean_ms=260, gct_sd_ms=8, vertical_ratio_mean=8.5, vertical_ratio_sd=0.4)

# Fresh run: minimal drift
rec_fresh = _make_run_records(200, stance_slope=0.1, cadence_cv_increase=1.0, vr_offset=0.0)
idx_fresh, detail_fresh = compute_fatigue_index(rec_fresh, baseline)
print(f'Fresh run:   fatigue_index={idx_fresh}  →  {detail_fresh}')

# Fatigued run: GCT increasing with distance, cadence breaking down, VR elevated
rec_tired = _make_run_records(200, stance_slope=4.0, cadence_cv_increase=2.5, vr_offset=0.8)
idx_tired, detail_tired = compute_fatigue_index(rec_tired, baseline)
print(f'Fatigued run: fatigue_index={idx_tired}  →  {detail_tired}')

# No baseline
idx_no_base, _ = compute_fatigue_index(rec_fresh, None)
print(f'No baseline: fatigue_index={idx_no_base}')

assert idx_fresh is not None and idx_tired is not None
assert idx_tired > idx_fresh, 'Fatigued run should have higher fatigue index'
assert idx_no_base is None

# Check modifier thresholds from spec
for idx, label in [(0.3, 'No modifier'), (0.7, '−10% duration'), (1.2, '−20% + cap Z2'), (1.8, '−25% + cap Z2 + alert')]:
    if idx is None: mod = 'None (no baseline)'
    elif idx < 0.5: mod = 'None'
    elif idx < 1.0: mod = 'duration × 0.90'
    elif idx < 1.5: mod = 'duration × 0.80; cap Z2'
    else:           mod = 'duration × 0.75; cap Z2; biomechanics alert'
    print(f'  fatigue_index={idx} → modifier: {mod} | label: {label}')

print('\nBiomechanics fatigue index: all checks passed')

Fresh run:   fatigue_index=0.152  →  {'stance_drift': np.float64(0.0004), 'cadence_cv_ratio': np.float64(1.455), 'vertical_ratio_z': np.float64(-0.076)}
Fatigued run: fatigue_index=0.529  →  {'stance_drift': np.float64(0.0154), 'cadence_cv_ratio': np.float64(1.456), 'vertical_ratio_z': np.float64(1.924)}
No baseline: fatigue_index=None
  fatigue_index=0.3 → modifier: None | label: No modifier
  fatigue_index=0.7 → modifier: duration × 0.90 | label: −10% duration
  fatigue_index=1.2 → modifier: duration × 0.80; cap Z2 | label: −20% + cap Z2
  fatigue_index=1.8 → modifier: duration × 0.75; cap Z2; biomechanics alert | label: −25% + cap Z2 + alert

Biomechanics fatigue index: all checks passed


## §4 — Safety Gates

In [41]:
def _gate(
    gate_id:           str,
    force_sport:       str | None,
    alert_tier:        str,
    message:           str,
    *,
    blocks:            bool  = False,
    max_zone:          int   = 5,
    max_strength_pct:  float = 1.0,
    volume_cap_pct:    float | None = None,
    volume_multiplier: float | None = None,
    no_strength:       bool  = False,
    disclaimer:        bool  = False,
) -> dict:
    return {
        'gate_id':          gate_id,
        'force_sport':      force_sport,
        'alert_tier':       alert_tier,
        'message':          message,
        'blocks':           blocks,
        'max_zone':         max_zone,
        'max_strength_pct': max_strength_pct,
        'volume_cap_pct':   volume_cap_pct,
        'volume_multiplier':volume_multiplier,
        'no_strength':      no_strength,
        'disclaimer':       disclaimer,
    }


def apply_safety_gates(s: dict) -> dict | None:
    acwr   = s.get('acwr')
    consec = s.get('hrv_consecutive_suppressed', 0)
    ramp   = s.get('ramp_rate')
    scores = s.get('readiness_scores') or {}   # guard against explicit None
    joint  = scores.get('joint_feel', 5)
    overall= scores.get('overall_feel', 5)
    comp_d = s.get('days_to_competition')
    comp_p = s.get('competition_priority')

    if acwr is not None and acwr > 1.8:
        return _gate('acwr_critical', 'rest', 'non_functional_overreaching_risk',
            "Acute load is >80% above chronic base. Rest or Z1 only.",
            disclaimer=True, blocks=True)

    if acwr is not None and acwr > 1.5:
        return _gate('acwr_danger', None, 'overreaching_caution',
            "Load ceiling reached. No intensity escalation today.",
            max_zone=2, max_strength_pct=0.70, blocks=False)

    if consec >= 5:
        return _gate('hrv_5d_suppressed', 'rest', 'consult_professional',
            "HRV suppressed 5+ consecutive days. Rest today.",
            disclaimer=True, blocks=True)

    if ramp is not None and ramp > 15:
        return _gate('ramp_rate_exceeded', None, 'ramp_rate_exceeded',
            f"CTL increased {ramp:.0f}% this week. Volume capped.",
            volume_cap_pct=100, blocks=False)

    if comp_d is not None and comp_p == 'A':
        if comp_d <= 2:
            return _gate('pre_comp_48h', None, 'taper',
                "A-race in 48h. No strength training. Z1–Z2 run only.",
                max_zone=2, no_strength=True, blocks=False)
        if comp_d <= 7:
            return _gate('pre_comp_week', None, 'taper',
                f"A-race in {comp_d} days. Volume at 60%.",
                volume_multiplier=0.60, blocks=False)

    if joint <= 2 or overall == 1:
        return _gate('injury_signal', 'mobility', 'injury_flag',
            "Joint discomfort flagged. High-impact activity not recommended.",
            disclaimer=True, blocks=True)

    return None


# ── Gate tests ─────────────────────────────────────────────────────────────────
def base_signals(**overrides):
    """Clean athlete state — no gates should fire."""
    s = dict(acwr=1.0, hrv_consecutive_suppressed=0, ramp_rate=5.0,
              readiness_scores={'joint_feel': 7, 'overall_feel': 7},
              days_to_competition=None, competition_priority=None)
    s.update(overrides)
    return s


gate_tests = [
    ('No gate fires (healthy)',        base_signals(),                                      None),
    ('Gate 0: ACWR critical (1.81)',   base_signals(acwr=1.81),                             'acwr_critical'),
    ('Gate 0: ACWR exactly 1.8',       base_signals(acwr=1.80),                             'acwr_danger'),   # not >1.8
    ('Gate 1: ACWR danger (1.6)',      base_signals(acwr=1.6),                              'acwr_danger'),
    ('Gate 2: HRV 5d suppressed',      base_signals(hrv_consecutive_suppressed=5),          'hrv_5d_suppressed'),
    ('Gate 2: HRV 4d suppressed',      base_signals(hrv_consecutive_suppressed=4),          None),  # not 5 yet
    ('Gate 3: Ramp rate 16%',          base_signals(ramp_rate=16),                          'ramp_rate_exceeded'),
    ('Gate 3: Ramp rate 15%',          base_signals(ramp_rate=15.0),                        None),  # not >15
    ('Gate 4: A-race in 2 days',       base_signals(days_to_competition=2, competition_priority='A'), 'pre_comp_48h'),
    ('Gate 4: A-race in 5 days',       base_signals(days_to_competition=5, competition_priority='A'), 'pre_comp_week'),
    ('Gate 4: B-race in 2 days',       base_signals(days_to_competition=2, competition_priority='B'), None),  # B-race: no gate
    ('Gate 5: Joint feel = 2',         base_signals(readiness_scores={'joint_feel':2,'overall_feel':7}), 'injury_signal'),
    ('Gate 5: Overall feel = 1',       base_signals(readiness_scores={'joint_feel':7,'overall_feel':1}), 'injury_signal'),
    ('Gate 5: readiness_scores=None',  base_signals(readiness_scores=None),                 None),   # no scores → no injury gate
    ('Gate ordering: ACWR > HRV',      base_signals(acwr=1.81, hrv_consecutive_suppressed=5), 'acwr_critical'),  # Gate 0 wins
]

print('Safety gate tests:')
all_pass = True
for name, signals, expected_gate in gate_tests:
    result = apply_safety_gates(signals)
    got    = result['gate_id'] if result else None
    ok     = '✓' if got == expected_gate else '✗'
    if got != expected_gate:
        all_pass = False
    print(f'  {ok} {name:<45} → {str(got):<22} (expected {expected_gate})')

if not all_pass:
    flag_gap('4', 'One or more gate tests failed — review boundary conditions', 'major')
else:
    print('\nAll gate tests passed')

Safety gate tests:
  ✓ No gate fires (healthy)                       → None                   (expected None)
  ✓ Gate 0: ACWR critical (1.81)                  → acwr_critical          (expected acwr_critical)
  ✓ Gate 0: ACWR exactly 1.8                      → acwr_danger            (expected acwr_danger)
  ✓ Gate 1: ACWR danger (1.6)                     → acwr_danger            (expected acwr_danger)
  ✓ Gate 2: HRV 5d suppressed                     → hrv_5d_suppressed      (expected hrv_5d_suppressed)
  ✓ Gate 2: HRV 4d suppressed                     → None                   (expected None)
  ✓ Gate 3: Ramp rate 16%                         → ramp_rate_exceeded     (expected ramp_rate_exceeded)
  ✓ Gate 3: Ramp rate 15%                         → None                   (expected None)
  ✓ Gate 4: A-race in 2 days                      → pre_comp_48h           (expected pre_comp_48h)
  ✓ Gate 4: A-race in 5 days                      → pre_comp_week          (expected pre_comp_week)
  ✓ 

## §5.1 — Readiness Aggregation

In [ ]:
def aggregate_readiness(
    hrv_status:       str,
    sleep_readiness:  str,
    daily_readiness:  dict | None,
    tsb:              float | None,
) -> str:  # 'peak' | 'good' | 'moderate' | 'low' | 'rest'
    score = 0
    score += {'elevated':4,'normal':2,'suppressed':0,'very_suppressed':-2,'no_data':1}[hrv_status]
    score += {'high':2,'moderate':1,'low':-1,'no_data':0}[sleep_readiness]
    if daily_readiness:
        # Expects 1-5 scale. build_recommendation normalises from 1-10 before calling.
        subj  = mean([daily_readiness.get('overall_feel', 3),
                      daily_readiness.get('legs_feel', 3),
                      daily_readiness.get('upper_feel', 3)])
        score += round((subj - 3) * 0.8)
    if tsb is not None:
        score += (1 if tsb > 15 else -1 if tsb < -20 else 0)
    return ('peak' if score >= 5 else 'good' if score >= 2 else
            'moderate' if score >= 0 else 'low' if score >= -2 else 'rest')


def normalise_readiness(scores: dict | None) -> dict | None:
    """Map 1-10 DB scores to the 1-5 scale aggregate_readiness expects."""
    if scores is None:
        return None
    return {k: (v - 1) / 9 * 4 + 1 for k, v in scores.items()}


# Unit tests — inputs on 1-5 scale (post-normalisation).
# Score thresholds: peak≥5, good≥2, moderate≥0, low≥-2, rest<-2.
# Key: very_suppressed(-2) + low(-1) = -3 → rest, not low.
# 'low' requires a milder combination that lands in [-2, -1].
readiness_tests = [
    ('Peak: elevated HRV + great sleep + high TSB',
     dict(hrv_status='elevated', sleep_readiness='high',
          daily_readiness={'overall_feel':5,'legs_feel':5,'upper_feel':5}, tsb=20),
     'peak'),    # 4+2+2+1=9
    ('Good: normal HRV + moderate sleep, neutral subjective',
     dict(hrv_status='normal', sleep_readiness='moderate',
          daily_readiness={'overall_feel':3,'legs_feel':3,'upper_feel':3}, tsb=5),
     'good'),    # 2+1+0+0=3
    ('Moderate: suppressed HRV + moderate sleep, neutral subjective',
     dict(hrv_status='suppressed', sleep_readiness='moderate',
          daily_readiness={'overall_feel':3,'legs_feel':3,'upper_feel':3}, tsb=-5),
     'moderate'), # 0+1+0+0=1
    ('Low: very suppressed HRV + moderate sleep',
     dict(hrv_status='very_suppressed', sleep_readiness='moderate',
          daily_readiness={'overall_feel':3,'legs_feel':3,'upper_feel':3}, tsb=-5),
     'low'),     # -2+1+0+0=-1
    ('Rest: very suppressed + poor sleep + deeply negative TSB',
     dict(hrv_status='very_suppressed', sleep_readiness='low',
          daily_readiness={'overall_feel':1,'legs_feel':1,'upper_feel':1}, tsb=-25),
     'rest'),    # -2-1-2-1=-6
    ('No data cold start',
     dict(hrv_status='no_data', sleep_readiness='no_data',
          daily_readiness=None, tsb=0),
     'moderate'), # 1+0+0+0=1
]

print('Readiness aggregation tests (1-5 scale inputs, post-normalisation):')
all_pass = True
for name, kwargs, expected in readiness_tests:
    result = aggregate_readiness(**kwargs)
    ok = '✓' if result == expected else '✗'
    if result != expected:
        all_pass = False
    print(f'  {ok} {name:<55} → {result:<10} (expected {expected})')

if all_pass:
    print('\nReadiness aggregation: all checks passed')
else:
    flag_gap('5.1', 'Readiness aggregation mismatch — check score thresholds', 'major')

## §5.3 — Concurrent Training Blocks

In [43]:
def build_concurrent_blocks(
    todays_sessions:    list[dict],
    yesterdays_session: dict | None,
    goal:               str,
) -> dict[str, str]:
    blocks = {}
    now    = datetime.utcnow()
    for s in todays_sessions:
        h_ago = (now - s['start_time']).total_seconds() / 3600
        if s['sport'] in ('running', 'cycling', 'swimming', 'trail_run'):
            if h_ago < 3.0 and goal in ('athlete', 'strength'):
                blocks['gym_power'] = (
                    f"Aerobic session {h_ago:.1f}h ago — explosive strength "
                    "attenuated within 3h. Substitute: hypertrophy or endurance work."
                )
    if yesterdays_session:
        yt = yesterdays_session.get('session_type')
        if yt == 'lower':
            blocks['run']       = "Lower gym yesterday — leg recovery priority"
            blocks['trail_run'] = "Lower gym yesterday — leg recovery priority"
            blocks['gym_lower'] = "Lower session yesterday — 48h required between lower sessions"
        if yt == 'upper':
            blocks['climb']     = "Upper gym yesterday — shoulder/elbow recovery"
            blocks['gym_upper'] = "Upper session yesterday — 48h required between upper sessions"
    return blocks


now = datetime.utcnow()
block_tests = [
    ('Run 1h ago + athlete goal → gym_power blocked',
     dict(todays_sessions=[{'sport':'running','start_time': now - timedelta(hours=1)}],
          yesterdays_session=None, goal='athlete'),
     {'gym_power'}),
    ('Run 4h ago → no block (interference resolved)',
     dict(todays_sessions=[{'sport':'running','start_time': now - timedelta(hours=4)}],
          yesterdays_session=None, goal='athlete'),
     set()),
    ('Lower gym yesterday → run + gym_lower blocked',
     dict(todays_sessions=[], yesterdays_session={'session_type':'lower'}, goal='athlete'),
     {'run', 'trail_run', 'gym_lower'}),
    ('Upper gym yesterday → climb + gym_upper blocked',
     dict(todays_sessions=[], yesterdays_session={'session_type':'upper'}, goal='athlete'),
     {'climb', 'gym_upper'}),
    ('Run 1h ago + hypertrophy goal → no power block',
     dict(todays_sessions=[{'sport':'running','start_time': now - timedelta(hours=1)}],
          yesterdays_session=None, goal='hypertrophy'),
     set()),
]

print('Concurrent block tests:')
all_pass = True
for name, kwargs, expected_keys in block_tests:
    result     = build_concurrent_blocks(**kwargs)
    result_keys = set(result.keys())
    ok         = '✓' if result_keys == expected_keys else '✗'
    if result_keys != expected_keys:
        all_pass = False
    print(f'  {ok} {name}')
    if result_keys != expected_keys:
        print(f'    got: {result_keys}  expected: {expected_keys}')

if not all_pass:
    flag_gap('5.3', 'Concurrent block logic mismatch — review 3h window and yesterday block rules', 'major')

Concurrent block tests:
  ✓ Run 1h ago + athlete goal → gym_power blocked
  ✓ Run 4h ago → no block (interference resolved)
  ✓ Lower gym yesterday → run + gym_lower blocked
  ✓ Upper gym yesterday → climb + gym_upper blocked
  ✓ Run 1h ago + hypertrophy goal → no power block


## §8 — Running Prescription

In [44]:
ZONE_DURATION_CAPS = {1: 60, 2: 90, 3: 60, 4: 50, 5: 40}
ZONE_LABELS        = {1:'active_recovery', 2:'aerobic_base',
                      3:'aerobic_threshold', 4:'threshold', 5:'vo2max'}

def prescribe_run(
    readiness_level:       str,
    tsb:                   float | None,
    gate_max_zone:         int   | None,
    time_available_min:    int,
    terrain:               str,
    zone_speeds:           dict,
    fatigue_index:         float | None,
    hr_decoupling_recent:  float | None,
) -> dict:
    base = {'peak':4,'good':3,'moderate':2,'low':1,'rest':1}[readiness_level]
    zone = base
    if tsb is not None and tsb < -5 and zone >= 3:
        zone = 2
    if gate_max_zone is not None:
        zone = min(zone, gate_max_zone)
    if fatigue_index is not None and fatigue_index > 1.0:
        zone = min(zone, 2)
    if hr_decoupling_recent is not None and hr_decoupling_recent > 5.0:
        zone = min(zone, 2)
    duration   = min(time_available_min, ZONE_DURATION_CAPS[zone])
    pace_range = zone_speeds.get(terrain, {}).get(f'z{zone}')
    return {'zone': zone, 'duration_min': duration, 'pace_range': pace_range,
            'session_type': ZONE_LABELS[zone], 'terrain': terrain}


example_zone_speeds = {'road': {'z1':'7:00–8:00', 'z2':'5:45–6:30', 'z3':'5:00–5:30',
                                 'z4':'4:30–4:55', 'z5':'4:00–4:25'},
                        'trail': {'z1':'8:00–9:30', 'z2':'6:30–7:30', 'z3':'5:45–6:15'}}

run_tests = [
    ('Peak readiness, good TSB → Z4 threshold',
     dict(readiness_level='peak', tsb=10, gate_max_zone=None, time_available_min=60,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=0.3, hr_decoupling_recent=2.0), 4),
    ('Peak readiness but TSB = -10 → Z2 (TSB overrides)',
     dict(readiness_level='peak', tsb=-10, gate_max_zone=None, time_available_min=60,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=None, hr_decoupling_recent=None), 2),
    ('Good readiness + ACWR gate max Z2',
     dict(readiness_level='good', tsb=5, gate_max_zone=2, time_available_min=90,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=None, hr_decoupling_recent=None), 2),
    ('High fatigue index (1.3) caps at Z2',
     dict(readiness_level='good', tsb=8, gate_max_zone=None, time_available_min=60,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=1.3, hr_decoupling_recent=None), 2),
    ('HR decoupling > 5% caps at Z2',
     dict(readiness_level='good', tsb=8, gate_max_zone=None, time_available_min=60,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=None, hr_decoupling_recent=6.5), 2),
    ('Time cap: peak but only 30 min available',
     dict(readiness_level='peak', tsb=15, gate_max_zone=None, time_available_min=30,
          terrain='road', zone_speeds=example_zone_speeds,
          fatigue_index=None, hr_decoupling_recent=None), 4),  # zone=4, duration=30 (capped by time)
]

print('Running prescription tests:')
all_pass = True
for name, kwargs, expected_zone in run_tests:
    result = prescribe_run(**kwargs)
    ok = '✓' if result['zone'] == expected_zone else '✗'
    if result['zone'] != expected_zone:
        all_pass = False
    print(f'  {ok} {name}')
    print(f'    zone={result["zone"]} (expected {expected_zone}), duration={result["duration_min"]}min, '
          f'type={result["session_type"]}, pace={result["pace_range"]}')

if not all_pass:
    flag_gap('8', 'Running prescription zone logic mismatch', 'major')

Running prescription tests:
  ✓ Peak readiness, good TSB → Z4 threshold
    zone=4 (expected 4), duration=50min, type=threshold, pace=4:30–4:55
  ✓ Peak readiness but TSB = -10 → Z2 (TSB overrides)
    zone=2 (expected 2), duration=60min, type=aerobic_base, pace=5:45–6:30
  ✓ Good readiness + ACWR gate max Z2
    zone=2 (expected 2), duration=90min, type=aerobic_base, pace=5:45–6:30
  ✓ High fatigue index (1.3) caps at Z2
    zone=2 (expected 2), duration=60min, type=aerobic_base, pace=5:45–6:30
  ✓ HR decoupling > 5% caps at Z2
    zone=2 (expected 2), duration=60min, type=aerobic_base, pace=5:45–6:30
  ✓ Time cap: peak but only 30 min available
    zone=4 (expected 4), duration=30min, type=threshold, pace=4:30–4:55


## §7 — Strength Prescription

In [45]:
LOAD_PCT = {
    'athlete':     {'peak':(0.80,0.85),'good':(0.75,0.80),'moderate':(0.65,0.70),'low':(0.55,0.65),'rest':(0.50,0.60)},
    'strength':    {'peak':(0.90,0.95),'good':(0.85,0.90),'moderate':(0.75,0.80),'low':(0.65,0.75),'rest':(0.60,0.65)},
    'hypertrophy': {'peak':(0.75,0.80),'good':(0.70,0.75),'moderate':(0.60,0.65),'low':(0.50,0.60),'rest':(0.45,0.55)},
}
REST_S = {'athlete': 150, 'strength': 240, 'hypertrophy': 75}

def prescribe_load(
    estimated_1rm:    float | None,
    goal:             str,
    readiness_level:  str,
) -> dict:
    if estimated_1rm is None:
        return {'load_kg_range': None, 'load_pct_1rm': None,
                'note': 'No 1RM data — work to a challenging weight for the rep range'}
    lo, hi = LOAD_PCT[goal][readiness_level]
    return {
        'load_kg_range': [round(estimated_1rm * lo, 1), round(estimated_1rm * hi, 1)],
        'load_pct_1rm':  f'{round(lo*100)}–{round(hi*100)}%',
        'rest_s':         REST_S[goal],
        'note':          '',
    }

def autoregulate_volume(base_sets: int, base_reps: int, load_feel: int | None) -> tuple:
    if load_feel is None:
        return base_sets, base_reps
    factor = {-2:1.20, -1:1.10, 0:1.00, 1:0.90, 2:0.80}[max(-2, min(2, load_feel))]
    return max(1, round(base_sets * factor)), max(4, round(base_reps * factor))


print('Strength prescription tests:')
# Squat 1RM = 100kg, peak readiness, athlete goal → 80–85kg
p = prescribe_load(100, 'athlete', 'peak')
print(f'  Squat 100kg 1RM, athlete, peak: {p["load_kg_range"]} @ {p["load_pct_1rm"]}')
assert p['load_kg_range'] == [80.0, 85.0]

# No 1RM data
p2 = prescribe_load(None, 'strength', 'good')
print(f'  No 1RM: {p2["note"]}')
assert p2['load_kg_range'] is None

print()
print('Autoregulation tests:')
for load_feel, desc in [(-2,'much too easy'), (-1,'slightly easy'), (0,'about right'), (1,'slightly hard'), (2,'too hard')]:
    s, r = autoregulate_volume(3, 8, load_feel)
    print(f'  load_feel={load_feel:+d} ({desc:<15}): {3} sets×{8} reps → {s} sets×{r} reps')

# Verify direction
s_easy, r_easy = autoregulate_volume(3, 8, -2)
s_hard, r_hard = autoregulate_volume(3, 8, 2)
assert s_easy > s_hard and r_easy > r_hard, 'Easy should produce more volume than hard'
print('Strength prescription: all checks passed')

Strength prescription tests:
  Squat 100kg 1RM, athlete, peak: [80.0, 85.0] @ 80–85%
  No 1RM: No 1RM data — work to a challenging weight for the rep range

Autoregulation tests:
  load_feel=-2 (much too easy  ): 3 sets×8 reps → 4 sets×10 reps
  load_feel=-1 (slightly easy  ): 3 sets×8 reps → 3 sets×9 reps
  load_feel=+0 (about right    ): 3 sets×8 reps → 3 sets×8 reps
  load_feel=+1 (slightly hard  ): 3 sets×8 reps → 3 sets×7 reps
  load_feel=+2 (too hard       ): 3 sets×8 reps → 2 sets×6 reps
Strength prescription: all checks passed


## Full Pipeline — build_recommendation()

In [46]:
def build_recommendation(signals: dict) -> dict:
    """
    Full pipeline as per spec §3–§8.
    signals keys: personal_hrv_mean, personal_hrv_sd, hrv_series,
                  acwr, sleep_readiness, readiness_scores (1-10 DB scale),
                  ctl, tsb, atl, ramp_rate,
                  days_to_competition, competition_priority, fatigue_index,
                  hr_decoupling_recent, goal, time_available_min,
                  terrain, zone_speeds, estimated_1rm, load_feel,
                  yesterdays_session, todays_sessions
    """
    today = signals.get('date', date.today())

    # 0. HRV status from per-athlete baseline stored in user_profile (§3.4)
    hrv_series = signals.get('hrv_series', [])
    p_mean     = signals.get('personal_hrv_mean')
    p_sd       = signals.get('personal_hrv_sd')
    if hrv_series and p_mean is not None and p_sd is not None:
        hrv_result = compute_hrv_status(hrv_series, p_mean, p_sd)
        signals = {**signals,
                   'hrv_status':                hrv_result['status'],
                   'hrv_z':                     hrv_result['z'],
                   'hrv_consecutive_suppressed': hrv_result['consecutive_suppressed']}

    # 1. Safety gates (§4) — first-wins
    gate = apply_safety_gates(signals)

    # 2. Readiness aggregation (§5.1)
    #    DB stores readiness fields on 1-10; normalise to 1-5 before formula.
    readiness = aggregate_readiness(
        hrv_status      = signals.get('hrv_status', 'no_data'),
        sleep_readiness = signals.get('sleep_readiness', 'no_data'),
        daily_readiness = normalise_readiness(signals.get('readiness_scores')),
        tsb             = signals.get('tsb'),
    )
    if gate and gate['blocks']:
        readiness = 'rest'

    # 3. Concurrent blocks (§5.3)
    blocks = build_concurrent_blocks(
        todays_sessions    = signals.get('todays_sessions', []),
        yesterdays_session = signals.get('yesterdays_session'),
        goal               = signals.get('goal', 'athlete'),
    )

    # 4. Running prescription (§8)
    run_rec = prescribe_run(
        readiness_level      = readiness,
        tsb                  = signals.get('tsb'),
        gate_max_zone        = gate['max_zone'] if gate and gate['max_zone'] < 5 else None,
        time_available_min   = signals.get('time_available_min', 60),
        terrain              = signals.get('terrain', 'road'),
        zone_speeds          = signals.get('zone_speeds', {}),
        fatigue_index        = signals.get('fatigue_index'),
        hr_decoupling_recent = signals.get('hr_decoupling_recent'),
    )

    # 5. Strength prescription (§7)
    strength_included = not (gate and gate['no_strength'])
    if strength_included and readiness not in ('rest',):
        load_feel  = signals.get('load_feel')
        str_goal   = signals.get('goal', 'athlete')
        base_sets, base_reps = {'peak':(4,6),'good':(3,8),'moderate':(3,10),'low':(2,12),'rest':(0,0)}[readiness]
        adj_sets, adj_reps   = autoregulate_volume(base_sets, base_reps, load_feel)
        # §9: timing — if an aerobic session ended < 3h ago, flag the interference window
        timing = 'anytime'
        now    = datetime.utcnow()
        for sess in signals.get('todays_sessions', []):
            if sess['sport'] in ('running', 'cycling', 'swimming', 'trail_run'):
                h_ago = (now - sess['start_time']).total_seconds() / 3600
                if h_ago < 3.0:
                    timing = 'after_run_3h_min'
                    break
        strength_rec = {
            'include': True,
            'sets':    adj_sets,
            'reps':    adj_reps,
            'load':    prescribe_load(signals.get('estimated_1rm'), str_goal, readiness),
            'timing':  timing,
        }
    else:
        strength_rec = {'include': False, 'sets': None, 'reps': None, 'load': None, 'timing': None}

    # 6. Confidence — degrades with missing signals
    n_present = sum(1 for k in ['hrv_status','sleep_readiness','readiness_scores','acwr','tsb','fatigue_index']
                    if signals.get(k) is not None and signals.get(k) != 'no_data')
    confidence = round(n_present / 6, 2)

    alerts = [{'tier': gate['alert_tier'], 'message': gate['message']}] if gate else []

    return {
        'date':            str(today),
        'readiness_level': readiness,
        'confidence':      confidence,
        'recommendation':  run_rec,
        'strength_block':  strength_rec,
        'blocks':          blocks,
        'alerts':          alerts,
        'gate_fired':      gate['gate_id'] if gate else None,
        'disclaimer':      STANDARD_DISCLAIMER if (gate and gate['disclaimer']) else None,
    }


print('Pipeline assembly: OK')
print('Testing with a sample state (elevated HRV +1.3 SD, strong subjective scores)...')
_sample_hrv = _make_hrv_series(PM, PSD, 30, [PM + 1.3 * PSD])
sample = build_recommendation({
    'personal_hrv_mean': PM, 'personal_hrv_sd': PSD, 'hrv_series': _sample_hrv,
    'acwr': 1.1, 'sleep_readiness': 'high',
    'readiness_scores': {'overall_feel':8,'legs_feel':8,'upper_feel':7,'joint_feel':8},  # 1-10 scale
    'tsb': 12, 'atl': 40, 'ctl': 52, 'ramp_rate': 4,
    'fatigue_index': 0.3, 'hr_decoupling_recent': 2.1,
    'goal': 'athlete', 'time_available_min': 75, 'terrain': 'road',
    'zone_speeds': example_zone_speeds, 'estimated_1rm': 100,
    'load_feel': 0, 'yesterdays_session': None, 'todays_sessions': [],
    'days_to_competition': None, 'competition_priority': None,
})
import json
print(json.dumps(sample, indent=2))

Pipeline assembly: OK
Testing with a sample state (elevated HRV +1.3 SD, strong subjective scores)...
{
  "date": "2026-05-13",
  "readiness_level": "peak",
  "confidence": 1.0,
  "recommendation": {
    "zone": 4,
    "duration_min": 50,
    "pace_range": "4:30\u20134:55",
    "session_type": "threshold",
    "terrain": "road"
  },
  "strength_block": {
    "include": true,
    "sets": 4,
    "reps": 6,
    "load": {
      "load_kg_range": [
        80.0,
        85.0
      ],
      "load_pct_1rm": "80\u201385%",
      "rest_s": 150,
      "note": ""
    },
    "timing": "anytime"
  },
  "blocks": {},
  "alerts": [],
  "gate_fired": null,
  "disclaimer": null
}


## Synthetic Scenario Suite

Each scenario represents a realistic athlete state. For each, the engine output is shown
alongside the expected call from the spec. Mismatches are flagged.

In [47]:
def make_hrv_series(status: str, consec: int = 0) -> list[float]:
    today_offset = {'elevated': +1.3, 'normal': 0.0,
                    'suppressed': -1.26, 'very_suppressed': -2.0}[status]
    return _make_hrv_series(PM, PSD, 30, [PM + today_offset * PSD],
                            consec_suppressed=consec)


def s(**overrides) -> dict:
    """Healthy baseline athlete — no gates, good readiness."""
    base = {
        'personal_hrv_mean': PM,
        'personal_hrv_sd':   PSD,
        'hrv_series':        make_hrv_series('normal'),
        'acwr': 1.0, 'ramp_rate': 5.0, 'tsb': 2, 'atl': 45, 'ctl': 47,
        'sleep_readiness': 'moderate',
        # 1-10 DB scale — pipeline normalises to 1-5 before aggregate_readiness.
        # 5/10 ≈ neutral, 7/10 = good joints (well clear of the ≤2 injury gate).
        'readiness_scores': {'overall_feel':5,'legs_feel':5,'upper_feel':5,'joint_feel':7},
        'fatigue_index': None, 'hr_decoupling_recent': None,
        'goal': 'athlete', 'time_available_min': 60, 'terrain': 'road',
        'zone_speeds': example_zone_speeds, 'estimated_1rm': 100,
        'load_feel': 0, 'yesterdays_session': None, 'todays_sessions': [],
        'days_to_competition': None, 'competition_priority': None,
    }
    base.update(overrides)
    return base


SCENARIOS = [
    {
        'name': '01 — Overtrained: ACWR critical',
        'signals': s(acwr=1.85, tsb=-22, sleep_readiness='low',
                     hrv_series=make_hrv_series('very_suppressed', consec=4)),
        'expect': {'gate_fired': 'acwr_critical', 'readiness_level': 'rest'},
    },
    {
        'name': '02 — Load caution: ACWR danger → zone capped at 2',
        'signals': s(acwr=1.6, tsb=-12,
                     hrv_series=make_hrv_series('suppressed')),
        'expect': {'gate_fired': 'acwr_danger', 'recommendation.zone': 2},
    },
    {
        'name': '03 — HRV 5-day suppression → forced rest',
        'signals': s(acwr=1.1,
                     hrv_series=make_hrv_series('suppressed', consec=5)),
        'expect': {'gate_fired': 'hrv_5d_suppressed', 'readiness_level': 'rest'},
    },
    {
        'name': '04 — A-race in 48h: no strength, Z2 max',
        'signals': s(days_to_competition=2, competition_priority='A'),
        'expect': {'gate_fired': 'pre_comp_48h', 'strength_block.include': False},
    },
    {
        'name': '05 — Peak day: quality session Z4',
        'signals': s(tsb=18, sleep_readiness='high',
                     hrv_series=make_hrv_series('elevated'),
                     readiness_scores={'overall_feel':9,'legs_feel':9,'upper_feel':9,'joint_feel':9}),
        'expect': {'gate_fired': None, 'readiness_level': 'peak', 'recommendation.zone': 4},
    },
    {
        'name': '06 — Good fitness but high fatigue index → forced Z2',
        'signals': s(fatigue_index=1.4),
        'expect': {'recommendation.zone': 2},
    },
    {
        'name': '07 — Lower gym yesterday → run blocked',
        'signals': s(yesterdays_session={'session_type':'lower'}),
        'expect': {'blocks.run': True},
    },
    {
        'name': '08 — Ran 1h ago, athlete goal → power gym blocked + timing flagged',
        'signals': s(todays_sessions=[{'sport':'running',
                                        'start_time': datetime.utcnow()-timedelta(hours=1)}]),
        'expect': {'blocks.gym_power': True},
    },
    {
        'name': '09 — No data cold start → moderate, no gate',
        'signals': s(personal_hrv_mean=None, personal_hrv_sd=None, hrv_series=[],
                     sleep_readiness='no_data', acwr=None, tsb=None,
                     readiness_scores=None, fatigue_index=None),
        'expect': {'gate_fired': None, 'readiness_level': 'moderate'},
    },
    {
        'name': '10 — Ramp rate exceeded (16%) → volume capped',
        'signals': s(ramp_rate=16),
        'expect': {'gate_fired': 'ramp_rate_exceeded'},
    },
    {
        'name': '11 — Joint feel=2/10 → injury signal + disclaimer',
        'signals': s(readiness_scores={'overall_feel':5,'legs_feel':5,'upper_feel':5,'joint_feel':2}),
        'expect': {'gate_fired': 'injury_signal', 'disclaimer': True},
    },
    {
        'name': '12 — TSB -10, elevated HRV → still caps at Z2',
        'signals': s(tsb=-10, sleep_readiness='high',
                     hrv_series=make_hrv_series('elevated')),
        'expect': {'recommendation.zone': 2},
    },
]


def get_nested(result: dict, key: str):
    parts = key.split('.')
    v = result
    for p in parts:
        v = v.get(p) if isinstance(v, dict) else None
    return v

print('=' * 70)
print('SYNTHETIC SCENARIO SUITE')
print('=' * 70)
all_pass = True
for sc in SCENARIOS:
    result = build_recommendation(sc['signals'])
    fails  = []
    for key, expected in sc['expect'].items():
        if key.startswith('blocks.'):
            actual = key.split('.')[1] in result['blocks']
        elif key == 'disclaimer':
            actual = result['disclaimer'] is not None
        else:
            actual = get_nested(result, key)
        if actual != expected:
            fails.append(f'{key}: got {actual!r}, expected {expected!r}')
    ok = '✓' if not fails else '✗'
    if fails:
        all_pass = False
    r, sb = result['recommendation'], result['strength_block']
    print(f'\n{ok} {sc["name"]}')
    print(f'  readiness={result["readiness_level"]}  gate={result["gate_fired"]}  confidence={result["confidence"]}')
    print(f'  zone={r["zone"]}  duration={r["duration_min"]}min  type={r["session_type"]}')
    print(f'  strength={sb["include"]}  timing={sb["timing"]}  blocks={list(result["blocks"].keys())}')
    for f in fails:
        print(f'  FAIL: {f}')

print()
print('=' * 70)
print(f'RESULT: {"ALL PASSED" if all_pass else "SOME FAILED — see spec gaps below"}')
print('=' * 70)

SYNTHETIC SCENARIO SUITE

✓ 01 — Overtrained: ACWR critical
  readiness=rest  gate=acwr_critical  confidence=0.83
  zone=1  duration=60min  type=active_recovery
  strength=False  timing=None  blocks=[]

✓ 02 — Load caution: ACWR danger → zone capped at 2
  readiness=moderate  gate=acwr_danger  confidence=0.83
  zone=2  duration=60min  type=aerobic_base
  strength=True  timing=anytime  blocks=[]

✓ 03 — HRV 5-day suppression → forced rest
  readiness=rest  gate=hrv_5d_suppressed  confidence=0.83
  zone=1  duration=60min  type=active_recovery
  strength=False  timing=None  blocks=[]

✓ 04 — A-race in 48h: no strength, Z2 max
  readiness=good  gate=pre_comp_48h  confidence=0.83
  zone=2  duration=60min  type=aerobic_base
  strength=False  timing=None  blocks=[]

✓ 05 — Peak day: quality session Z4
  readiness=peak  gate=None  confidence=0.83
  zone=4  duration=50min  type=threshold
  strength=True  timing=anytime  blocks=[]

✓ 06 — Good fitness but high fatigue index → forced Z2
  readine

## Findings & Spec Gaps

Any gaps found during execution are collected here. Review before updating the spec.

In [ ]:
print('=' * 70)
print('SPEC GAP LOG')
print('=' * 70)

if not SPEC_GAPS:
    print('No gaps found — logic is consistent with spec.')
    print('Safe to send spec to dev team.')
else:
    df_gaps = pd.DataFrame(SPEC_GAPS)
    for _, row in df_gaps.iterrows():
        print(f'  [{row["severity"].upper()}] §{row["section"]}: {row["description"]}')
    print()
    major = df_gaps[df_gaps['severity'] == 'major']
    if len(major) > 0:
        print(f'⚠ {len(major)} MAJOR gap(s) — spec needs revision before dev hand-off')
    else:
        print(f'Minor gaps only — can proceed to dev with noted caveats')

print()
print('OPEN QUESTIONS for spec review:')
print('  1. LOAD_PCT table in §7.2 does not define readiness_level="rest" row.')
print('     Prototype added rest=(0.50,0.60) for athlete; spec should make this explicit.')
print('  2. build_recommendation() spec (§5) does not fully define how primary_sport')
print('     is selected when no gate fires and multiple sports are valid. The sport')
print('     selection matrix in §5.2 maps readiness → session type, not sport.')
print('     Needs a concrete sport ranking rule (user priority weights from user_profile).')
print('  3. confidence score is not defined in spec. Prototype uses signal availability')
print('     ratio. Spec should define this formally if frontend will display it.')